# Wikidata VGAE Training

Train the R-GCN-based variational graph autoencoder on the same `tkgl-smallpedia` Wikidata graph used by the R-GCN experiments. The notebook loads `R-GCN/checkpoints/wiki_graph.pkl` and saves training results to a pickle file.

In [ ]:
from pathlib import Path
import datetime as dt
import pickle
import random
import sys

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "R-GCN" / "rgcn_model.py").exists():
            return candidate
    raise RuntimeError("Could not find project root containing R-GCN/rgcn_model.py")


PROJECT_ROOT = find_project_root(Path.cwd())
RGCN_DIR = PROJECT_ROOT / "R-GCN"
CHECKPOINT_DIR = RGCN_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility for wiki_graph.pkl files saved from notebooks.
RelationalGraph = type("RelationalGraph", (), {})


class GraphCheckpointUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "__main__" and name == "RelationalGraph":
            return RelationalGraph
        return super().find_class(module, name)

sys.path.insert(0, str(RGCN_DIR))

from vgae_model import VGAE
from rgcn_model import sample_negatives

DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

In [ ]:
GRAPH_CKPT = RGCN_DIR / "checkpoints" / "wiki_graph.pkl"
RESULTS_PATH = CHECKPOINT_DIR / "vgae_wikidata_results.pkl"

if not GRAPH_CKPT.exists():
    raise FileNotFoundError(
        f"Missing {GRAPH_CKPT}. Run R-GCN/wikidata.ipynb first to build wiki_graph.pkl."
    )

with GRAPH_CKPT.open("rb") as f:
    wiki_graph = GraphCheckpointUnpickler(f).load()

print(f"Loaded graph: {wiki_graph.name}")
print(f"Nodes: {wiki_graph.num_nodes:,}")
print(f"Relations: {wiki_graph.num_relations:,}")
print(f"Train edges: {len(wiki_graph.train_edges):,}")
print(f"Val edges: {len(wiki_graph.val_edges):,}")
print(f"Test edges: {len(wiki_graph.test_edges):,}")
print(f"Node features: {wiki_graph.node_features is not None}")

In [ ]:
SEED = 42
HIDDEN_DIM = 64
LATENT_DIM = 64
EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 4096
NEG_RATIO = 1
KL_BETA = 0.1
EVAL_EVERY = 5
EVAL_POS_LIMIT = None
EVAL_NEGATIVES = 50
EVAL_BATCH_SIZE = 4096

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

config = {
    "seed": SEED,
    "hidden_dim": HIDDEN_DIM,
    "latent_dim": LATENT_DIM,
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "neg_ratio": NEG_RATIO,
    "kl_beta": KL_BETA,
    "eval_every": EVAL_EVERY,
    "eval_pos_limit": EVAL_POS_LIMIT,
    "eval_negatives": EVAL_NEGATIVES,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "device": str(DEVICE),
}
config

In [ ]:
@torch.no_grad()
def evaluate_vgae(
    model,
    graph,
    split_edges,
    edge_index,
    edge_type,
    *,
    num_negatives=50,
    max_pos=2000,
    eval_batch_size=4096,
):
    model.eval()
    split_edges = split_edges.to(DEVICE)
    if max_pos is not None and len(split_edges) > max_pos:
        idx = torch.randperm(len(split_edges), device=DEVICE)[:max_pos]
        split_edges = split_edges[idx]

    node_features = graph.node_features.to(DEVICE) if graph.node_features is not None else None
    z, _, _ = model(edge_index, edge_type, node_features=node_features, num_nodes=graph.num_nodes)

    reciprocal_ranks = []
    hits_at_1 = []
    hits_at_3 = []
    hits_at_10 = []

    for start in range(0, len(split_edges), eval_batch_size):
        batch = split_edges[start : start + eval_batch_size]
        src = batch[:, 0]
        dst = batch[:, 1]
        rel = batch[:, 2]
        pos_scores = model.decoder(z, src, dst, rel)

        neg_dst = torch.randint(0, graph.num_nodes, (len(batch), num_negatives), device=DEVICE)
        neg_scores = model.decoder(
            z,
            src[:, None].expand(-1, num_negatives).reshape(-1),
            neg_dst.reshape(-1),
            rel[:, None].expand(-1, num_negatives).reshape(-1),
        ).view(len(batch), num_negatives)

        ranks = ((neg_scores >= pos_scores[:, None]).sum(dim=1) + 1).float()
        reciprocal_ranks.append(1.0 / ranks)
        hits_at_1.append((ranks <= 1).float())
        hits_at_3.append((ranks <= 3).float())
        hits_at_10.append((ranks <= 10).float())

    reciprocal_ranks = torch.cat(reciprocal_ranks)
    hits_at_1 = torch.cat(hits_at_1)
    hits_at_3 = torch.cat(hits_at_3)
    hits_at_10 = torch.cat(hits_at_10)
    return {
        "num_pos": int(len(split_edges)),
        "mrr": float(reciprocal_ranks.mean().item()),
        "hits@1": float(hits_at_1.mean().item()),
        "hits@3": float(hits_at_3.mean().item()),
        "hits@10": float(hits_at_10.mean().item()),
    }


def train_vgae(model, graph):
    model.to(DEVICE)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    train_edges = graph.train_edges.to(DEVICE)
    edge_index = train_edges[:, :2].t().contiguous()
    edge_type = train_edges[:, 2]
    node_features = graph.node_features.to(DEVICE) if graph.node_features is not None else None

    history = {"loss": [], "bce_loss": [], "kl_loss": [], "val": []}

    for epoch in tqdm(range(1, EPOCHS + 1), desc="Training VGAE"):
        model.train()
        shuffled_edges = train_edges[torch.randperm(len(train_edges), device=DEVICE)]
        epoch_loss = 0.0
        epoch_bce = 0.0
        epoch_kl = 0.0
        num_batches = 0

        for start in range(0, len(shuffled_edges), BATCH_SIZE):
            pos_edges = shuffled_edges[start : start + BATCH_SIZE]
            neg_edges = sample_negatives(pos_edges, graph.num_nodes, NEG_RATIO)
            all_edges = torch.cat([pos_edges, neg_edges], dim=0)
            labels = torch.cat([
                torch.ones(len(pos_edges), device=DEVICE),
                torch.zeros(len(neg_edges), device=DEVICE),
            ])

            z, mu, log_var = model(
                edge_index,
                edge_type,
                node_features=node_features,
                num_nodes=graph.num_nodes,
            )

            logits = model.decoder(z, all_edges[:, 0], all_edges[:, 1], all_edges[:, 2])
            bce_loss = F.binary_cross_entropy_with_logits(logits, labels)
            kl_loss = -0.5 * torch.sum(
                1 + log_var - mu.pow(2) - log_var.exp(),
                dim=1,
            ).mean()
            loss = bce_loss + KL_BETA * kl_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += float(loss.item())
            epoch_bce += float(bce_loss.item())
            epoch_kl += float(kl_loss.item())
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        avg_bce = epoch_bce / num_batches
        avg_kl = epoch_kl / num_batches
        history["loss"].append(avg_loss)
        history["bce_loss"].append(avg_bce)
        history["kl_loss"].append(avg_kl)

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            val_metrics = evaluate_vgae(
                model,
                graph,
                graph.val_edges,
                edge_index,
                edge_type,
                num_negatives=EVAL_NEGATIVES,
                max_pos=EVAL_POS_LIMIT,
                eval_batch_size=EVAL_BATCH_SIZE,
            )
            val_metrics["epoch"] = epoch
            history["val"].append(val_metrics)
            print(
                f"Epoch {epoch:03d} | loss={avg_loss:.4f} "
                f"bce={avg_bce:.4f} kl={avg_kl:.4f} "
                f"val_mrr={val_metrics['mrr']:.4f} hits@10={val_metrics['hits@10']:.4f}"
            )
        else:
            print(
                f"Epoch {epoch:03d} | loss={avg_loss:.4f} "
                f"bce={avg_bce:.4f} kl={avg_kl:.4f}"
            )

    return history, edge_index, edge_type

In [ ]:
feat_dim = wiki_graph.node_features.shape[1] if wiki_graph.node_features is not None else None

model = VGAE(
    num_nodes=wiki_graph.num_nodes,
    hidden_dim=HIDDEN_DIM,
    num_relations=wiki_graph.num_relations,
    latent_dim=LATENT_DIM,
    feat_dim=feat_dim,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
history, train_edge_index, train_edge_type = train_vgae(model, wiki_graph)

In [ ]:
test_metrics = evaluate_vgae(
    model,
    wiki_graph,
    wiki_graph.test_edges,
    train_edge_index,
    train_edge_type,
    num_negatives=EVAL_NEGATIVES,
    max_pos=EVAL_POS_LIMIT,
    eval_batch_size=EVAL_BATCH_SIZE,
)

test_metrics

In [ ]:
model_cpu_state = {
    key: value.detach().cpu()
    for key, value in model.state_dict().items()
}

results = {
    "created_at": dt.datetime.now().isoformat(timespec="seconds"),
    "graph_checkpoint": str(GRAPH_CKPT),
    "graph": {
        "name": wiki_graph.name,
        "num_nodes": wiki_graph.num_nodes,
        "num_relations": wiki_graph.num_relations,
        "num_train_edges": int(len(wiki_graph.train_edges)),
        "num_val_edges": int(len(wiki_graph.val_edges)),
        "num_test_edges": int(len(wiki_graph.test_edges)),
        "has_node_features": wiki_graph.node_features is not None,
    },
    "config": config,
    "history": history,
    "test_metrics": test_metrics,
    "model_state_dict": model_cpu_state,
}

with RESULTS_PATH.open("wb") as f:
    pickle.dump(results, f)

print(f"Saved VGAE training results to {RESULTS_PATH}")